In [1]:
import pandas as pd
import numpy as np
import shutil
import os
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import (
    BertTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding, 
    EarlyStoppingCallback
)

In [ ]:
configuraciones = [
    {"nombre": "no_prep_len_256", "archivo": "../Data/raw/tcga_simple_train.csv", "max_len": 256},
    {"nombre": "no_prep_len_384", "archivo": "../Data/raw/tcga_simple_train.csv", "max_len": 384},
    {"nombre": "no_prep_len_512", "archivo": "../Data/raw/tcga_simple_train.csv", "max_len": 512},
    {"nombre": "prep_0_len_256", "archivo": "../Data/interim/tcga_simple_train_preprocessed_0.csv", "max_len": 256},
    {"nombre": "prep_0_len_384", "archivo": "../Data/interim/tcga_simple_train_preprocessed_0.csv", "max_len": 384},
    {"nombre": "prep_0_len_512", "archivo": "../Data/interim/tcga_simple_train_preprocessed_0.csv", "max_len": 512},
    {"nombre": "prep_1_len_256", "archivo": "../Data/interim/tcga_simple_train_preprocessed_1.csv", "max_len": 256},
    {"nombre": "prep_1_len_384", "archivo": "../Data/interim/tcga_simple_train_preprocessed_1.csv", "max_len": 384},
    {"nombre": "prep_1_len_512", "archivo": "../Data/interim/tcga_simple_train_preprocessed_1.csv", "max_len": 512},
    {"nombre": "prep_2_len_256", "archivo": "../Data/interim/tcga_simple_train_preprocessed_2.csv", "max_len": 256},
    {"nombre": "prep_2_len_384", "archivo": "../Data/interim/tcga_simple_train_preprocessed_2.csv", "max_len": 384},
    {"nombre": "prep_2_len_512", "archivo": "../Data/interim/tcga_simple_train_preprocessed_2.csv", "max_len": 512},
    {"nombre": "prep_3_len_256", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3.csv", "max_len": 256},
    {"nombre": "prep_3_len_384", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3.csv", "max_len": 384},
    {"nombre": "prep_3_len_512", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3.csv", "max_len": 512},
    {"nombre": "prep_4_len_256", "archivo": "../Data/interim/tcga_simple_train_preprocessed_4.csv", "max_len": 256},
    {"nombre": "prep_4_len_384", "archivo": "../Data/interim/tcga_simple_train_preprocessed_4.csv", "max_len": 384},
    {"nombre": "prep_4_len_512", "archivo": "../Data/interim/tcga_simple_train_preprocessed_4.csv", "max_len": 512},
]

In [3]:
label_mapping = {"T1": 0, "T2": 1, "T3": 2, "T4": 3}
tokenizer = BertTokenizer.from_pretrained("dmis-lab/biobert-v1.1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

resultados_finales = []

In [4]:
for conf in configuraciones:
    print("\n" + "="*50)
    print(f"TRABAJANDO EN: {conf['nombre']}")
    print("="*50)

    df = pd.read_csv(conf['archivo'])
    X_train, X_test, y_train, y_test = train_test_split(df['text'], df['t'], test_size=0.2, random_state=23, stratify=df['t'])
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=33, stratify=y_train)

    train_ds = Dataset.from_dict({"text": X_train.tolist(), "labels": [label_mapping[label] for label in y_train.tolist()]})
    val_ds = Dataset.from_dict({"text": X_val.tolist(), "labels": [label_mapping[label] for label in y_val.tolist()]})
    test_ds = Dataset.from_dict({"text": X_test.tolist(), "labels": [label_mapping[label] for label in y_test.tolist()]})

    def tokenize_fn(df):
        return tokenizer(df["text"], padding="max_length", truncation=True, max_length=conf['max_len'])

    train_tkn = train_ds.map(tokenize_fn, batched=True)
    val_tkn = val_ds.map(tokenize_fn, batched=True)
    test_tkn = test_ds.map(tokenize_fn, batched=True)

    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    weights = compute_class_weight("balanced", classes=np.unique(y_train_encoded), y=y_train_encoded)
    weights_tensor = torch.tensor(weights, dtype=torch.float)

    model = AutoModelForSequenceClassification.from_pretrained("dmis-lab/biobert-v1.1", num_labels=4)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.loss_function = torch.nn.CrossEntropyLoss(weight=weights_tensor.to(device))

    bs = 16 if conf['max_len'] <= 384 else 8 

    training_args = TrainingArguments(
        output_dir=f"../Models/biobert-{conf['nombre']}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs,
        num_train_epochs=20,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=True if torch.cuda.is_available() else False,
        logging_steps=200,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tkn,
        eval_dataset=val_tkn,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()

    metrics = trainer.evaluate(test_tkn)
    resultados_finales.append({
        "Escenario": conf['nombre'],
        "Longitud": conf['max_len'],
        "Macro-F1": metrics['eval_macro_f1']
    })

    del model
    del trainer
    shutil.rmtree("../Models")
    os.makedirs("../Models")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


TRABAJANDO EN: prep_4_len_512


Map:   0%|          | 0/3300 [00:00<?, ? examples/s]

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

Map:   0%|          | 0/1032 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1
0.200798,0.718500,9,0.775756


In [5]:
df_results = pd.DataFrame(resultados_finales)
print("\n--- COMPARATIVA FINAL ---")
print(df_results)

df_results.to_csv("../Results/Metrics/biobert_results.csv", index=False)


--- COMPARATIVA FINAL ---
        Escenario  Longitud  Macro-F1
0  prep_4_len_512       512  0.775756
